# Qwen3-8B × DRCD QLoRA 微調（Colab）

繁體中文抽取式閱讀理解 QLoRA 微調，對應 [PLAN.md](../PLAN.md) Phase 2（200 筆 sanity check）／Phase 3（全量訓練）。

## 執行前準備

1. **Colab 執行階段**：Runtime > Change runtime type > GPU（L4 為主，搶得到 A100 更好；免費層通常只有 T4，Colab Pro 才會分配到 L4）
2. **Colab Secrets**（左側工具列鑰匙圖示）：
   - `HF_TOKEN`：**write** 權限的 Hugging Face token（用來讀取 Phase 1 的 private dataset、推送 checkpoint）
   - `WANDB_API_KEY`：Weights & Biases API key
   - 兩個 secret 都要記得打開「Notebook access」開關，否則 `userdata.get()` 會報錯
3. **Google Drive**：本 notebook 會掛載你的 Drive 作為 checkpoint 的備援通道（HF Hub 是主要通道）
4. **設定檔**（可選）：若把 `configs/train_config.yaml` 上傳到 Colab 的工作目錄（跟本 notebook 同一層），會優先讀取外部檔案；
   否則使用 notebook 內建、數值完全相同的預設設定

## 斷線續訓設計（Colab Pro 沒有背景執行，session 隨時可能斷）

**這顆 notebook 設計成「斷線後重跑全部 cell 即可自動接續」**：每次執行都會先檢查 HF Hub（找不到再查 Google Drive）
有沒有前一次留下的 checkpoint，有的話就從那裡繼續訓練，global step、optimizer 狀態都會接上（optimizer 狀態的完整還原
在 PEFT/LoRA 場景下不保證 100%，屬已知風險，可接受）。

**Phase 2 驗收項目**：跑一次 sanity check → 訓練途中手動中斷（Runtime > Interrupt execution，或直接關分頁）
→ 重新執行全部 cell → 確認真的接著訓練、不是從頭開始。notebook 最後有一格專門說明這個演練怎麼做。


## 0. 下載相關環境變數（必須在這裡設，不能晚設）

**已知坑**（Phase 2 實測踩到、三路查證直接讀 huggingface_hub 原始碼確認）：huggingface_hub 新版下載大檔案預設
會走 **`hf_xet`**（取代舊版 `hf_transfer` 的高速傳輸元件），這套機制在 Colab 的網路環境會卡死在 0%、不會報錯、
也不會 timeout——小檔案（config、tokenizer 那些）走的是別條路徑所以正常，只有大的 safetensors 權重檔會卡住
（GitHub 上有多筆一模一樣的 2026 年回報，其中一筆就是 Colab 環境）。

`HF_HUB_ENABLE_HF_TRANSFER` 管的是舊機制，對 `hf_xet` 沒有作用；正確的開關是 `HF_HUB_DISABLE_XET`。
但查證也發現這個環境變數本身在某些版本**不保證生效**，所以保險做法是連 `hf_xet` 套件都直接移除
（see 下面安裝那格最後一行），環境變數留著當備援。

**這格必須是全 notebook 第一個真正執行的 code cell**：huggingface_hub 的環境變數是在被 import 的當下讀一次、
存成常數，之後才設定就沒用了。如果你是接續前面已經 import 過 huggingface_hub 的 session 重跑這格，
**先 Runtime > Restart session 再整份重跑**，不要只重跑這一格。


In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"        # 真正的關鍵：關掉會在 Colab 卡死的 hf_xet 傳輸機制
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # 舊機制，現在多半是 no-op，留著當保險不影響任何東西


## 1. GPU 偵測

In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
gpu_name = gpu_stats.name
total_mem_gb = round(gpu_stats.total_memory / 1024**3, 2)
print(f"分配到的 GPU：{gpu_name}（{total_mem_gb} GB）")

if "T4" in gpu_name:
    print("注意：分配到 T4，訓練會比 L4 慢（PLAN.md §2.4 預估以 L4 為基準），且 24GB->16GB VRAM 打八折要留意 OOM。")
elif "A100" in gpu_name:
    print("分配到 A100，訓練會比 L4 快，但消耗的 compute units 也更多（PLAN.md §2.4）。")
elif "L4" in gpu_name:
    print("分配到 L4，符合 PLAN.md 預估基準。")


## 2. 設定（config）

`configs/train_config.yaml` 是超參數的單一事實來源（PLAN.md §4.6）。這裡優先讀取上傳到 Colab 工作目錄的外部檔案，
找不到就用下面內建的預設值（與 yaml 內容保持一致，兩邊有任何調整都要同步改）。

**要切換 Phase 2 → Phase 3**：把 `mode` 從 `"sanity_check"` 改成 `"full_run"`（外部 yaml 或內建預設都可以改）。


In [ ]:
import os
import yaml

CONFIG_PATH = "train_config.yaml"

DEFAULT_CONFIG = {
    "mode": "full_run",
    "model": {
        "base_model": "unsloth/Qwen3-8B-unsloth-bnb-4bit",
        "chat_template": "qwen3",
        "max_length": 2048,
    },
    "lora": {
        "r": 16,
        "alpha": 16,
        "dropout": 0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "use_gradient_checkpointing": "unsloth",
        "random_state": 3407,
    },
    "sanity_check": {
        "max_train_samples": 200,
        "eval_holdout": 20,
        "num_train_epochs": 3,
        "save_steps": 5,
        "eval_steps": 5,
        "logging_steps": 1,
    },
    "full_run": {
        "max_train_samples": None,
        "eval_holdout": 200,
        "num_train_epochs": 2,
        "save_steps": 50,
        "eval_steps": 50,
        "logging_steps": 10,
    },
    "training": {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2.0e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "optim": "adamw_8bit",
        "bf16": True,
        "weight_decay": 0.001,
        "save_total_limit": 2,
        "packing": False,
        "seed": 3407,
    },
    "hub": {
        "dataset_repo": "steven0226/drcd-zhtw-extractive-qa-sft",
        "adapter_checkpoint_repo_sanity": "steven0226/qwen3-8b-drcd-qa-ckpt-sanity",
        "adapter_checkpoint_repo_full": "steven0226/qwen3-8b-drcd-qa-ckpt",
    },
    "wandb": {"project": "qwen3-drcd-qlora"},
    "drive": {"backup_dir": "/content/drive/MyDrive/qwen3-drcd-qlora-backup"},
}

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    print(f"讀取外部設定檔：{CONFIG_PATH}")
else:
    cfg = DEFAULT_CONFIG
    print("找不到外部設定檔，使用 notebook 內建預設值（與 configs/train_config.yaml 同步維護）")

MODE = cfg["mode"]
assert MODE in ("sanity_check", "full_run"), f"未知的 mode：{MODE}"
mode_cfg = cfg[MODE]
print(f"mode = {MODE}")
print(cfg)


## 3. 安裝套件

照 Unsloth 官方現行（2026-07）notebook 的安裝順序（查證於 `unslothai/notebooks` repo）：
先裝 Unsloth 全家桶（`--no-deps` 避免 pip resolver 亂跑版本），再釘死 `transformers`/`trl` 版本
（Unsloth 刻意固定這兩個版本以避免上游 API 變動打壞自家的 monkey-patch）。

最後移除 `hf_xet` 套件——這是本專案在 Colab 上實測會卡死大檔案下載的元凶（見上面「0. 下載相關環境變數」那格），
套件不存在的話 huggingface_hub 就一定會退回穩定的 HTTP 下載路徑，比只設環境變數更保險。


In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch as _torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(_torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install pyyaml wandb
!pip uninstall -y hf_xet -q


In [ ]:
# 安裝驗證：這格不加 %%capture，安裝若有問題會在這裡直接炸開，不會悄悄帶著壞環境往下跑
import unsloth, transformers, trl, peft, datasets, huggingface_hub, wandb

print("unsloth:", unsloth.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("wandb:", wandb.__version__)


## 4. Secrets 與登入

`WANDB_PROJECT` 等環境變數必須在 `SFTTrainer` 建立**之前**設好，晚了 W&B callback 讀不到（查證結果，見 PLAN.md 附錄）。


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_PROJECT"] = cfg["wandb"]["project"]

import wandb
wandb.login(key=WANDB_API_KEY)

print("HF_TOKEN 與 WANDB_API_KEY 已從 Colab Secrets 載入（不會印出內容）")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
# mode 隔離：sanity_check 跟 full_run 各自獨立的 Drive 備援目錄，
# 避免切換 mode 之後 full_run 誤續訓自 sanity_check 留下的小 checkpoint
DRIVE_BACKUP_DIR = os.path.join(cfg["drive"]["backup_dir"], MODE)
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"Drive 備援目錄：{DRIVE_BACKUP_DIR}")


## 5. Pre-flight：HF token write 權限煙霧測試

**先炸在這裡，不要炸在訓練 6 小時之後。**

查證發現一個真實的坑：`create_repo(..., exist_ok=True)` 在 repo 已存在、token 卻是唯讀（403）時會**靜默吞掉錯誤、回報成功**
（huggingface_hub#2419）。所以光呼叫 `create_repo` 不夠，一定要接一個真正的 `upload_file` 才能確實驗證 write 權限。


In [ ]:
from huggingface_hub import HfApi, whoami
from huggingface_hub.errors import HfHubHTTPError, LocalTokenNotFoundError


def preflight_check_hf_write_access(repo_id: str, token: str, repo_type: str = "model") -> None:
    try:
        who = whoami(token=token)
        print(f"HF whoami 通過：登入身分 {who.get('name')!r}")
    except (HfHubHTTPError, LocalTokenNotFoundError) as e:
        raise RuntimeError(
            "HF token 缺失或完全無效。請到 Colab Secrets 確認 HF_TOKEN，"
            "或到 https://huggingface.co/settings/tokens 產生新 token。"
        ) from e

    api = HfApi(token=token)
    try:
        api.create_repo(repo_id=repo_id, repo_type=repo_type, private=True, exist_ok=True)
        api.upload_file(
            path_or_fileobj=b"preflight-ok",
            path_in_repo=".smoke_test",
            repo_id=repo_id,
            repo_type=repo_type,
            commit_message="pre-flight write-access check",
        )
    except HfHubHTTPError as e:
        status = e.response.status_code if e.response is not None else "unknown"
        raise RuntimeError(
            f"HF token 沒有通過 write 煙霧測試（HTTP {status}）。"
            f"token 可能是唯讀或沒有 '{repo_id}' 的寫入權限。"
            f"請到 https://huggingface.co/settings/tokens 產生 write 權限的 token，"
            f"更新 Colab Secrets 後重跑這一格。"
        ) from e
    print(f"HF write 權限驗證通過：{repo_id}（訓練前的最後一道防線，安全了）")


HUB_MODEL_ID = cfg["hub"]["adapter_checkpoint_repo_sanity" if MODE == "sanity_check" else "adapter_checkpoint_repo_full"]
DATASET_REPO = cfg["hub"]["dataset_repo"]

preflight_check_hf_write_access(HUB_MODEL_ID, HF_TOKEN)


## 6. 續訓偵測

查 HF Hub 上這個 checkpoint repo 有沒有 `last-checkpoint/`（`hub_strategy="checkpoint"` 會把最新 checkpoint
推到這個固定名稱的子資料夾），以及 Google Drive 備援目錄有沒有對應的 checkpoint，**兩邊都有的話比較
`trainer_state.json` 裡的 `global_step`，選比較新的那份**——HF Hub 的推送是非同步的，斷線時機不巧的話
Drive 手上可能有比 Hub 更新的 checkpoint（例如剛存完本地檔案、Drive 也備份完了，但 Hub 那次 push 還沒送出就斷線），
如果無條件優先信任 Hub 會平白損失已經訓練完的進度。

**已知坑**（查證結果）：`trainer.train(resume_from_checkpoint=True)` 只會自動找 `checkpoint-<數字>` 這種命名的資料夾，
認不出 `last-checkpoint`，所以續訓一定要傳明確路徑字串，不能傳 `True`。


In [ ]:
import json as _json

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

api = HfApi(token=HF_TOKEN)


def _read_global_step(trainer_state_path):
    try:
        with open(trainer_state_path, encoding="utf-8") as f:
            return _json.load(f).get("global_step", -1)
    except Exception:
        return -1


candidates = []  # [(global_step, path, 來源標籤), ...]

try:
    has_ckpt = api.file_exists(
        repo_id=HUB_MODEL_ID, filename="last-checkpoint/trainer_state.json", token=HF_TOKEN
    )
except Exception as e:
    print(f"HF Hub checkpoint 存在性檢查失敗（{type(e).__name__}: {e}），視為沒有")
    has_ckpt = False

if has_ckpt:
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        allow_patterns=["last-checkpoint/*"],
        local_dir=OUTPUT_DIR,
        token=HF_TOKEN,
    )
    hub_candidate = os.path.join(OUTPUT_DIR, "last-checkpoint")
    hub_ts_path = os.path.join(hub_candidate, "trainer_state.json")
    if os.path.isdir(hub_candidate) and os.path.exists(hub_ts_path):
        candidates.append((_read_global_step(hub_ts_path), hub_candidate, "HF Hub"))

drive_candidate = os.path.join(DRIVE_BACKUP_DIR, "last-checkpoint")
drive_ts_path = os.path.join(drive_candidate, "trainer_state.json")
if os.path.isdir(drive_candidate) and os.path.exists(drive_ts_path):
    candidates.append((_read_global_step(drive_ts_path), drive_candidate, "Google Drive 備援"))

resume_path = None
if candidates:
    candidates.sort(key=lambda c: c[0], reverse=True)
    best_step, resume_path, source = candidates[0]
    print(f"偵測到可續訓的 checkpoint（來源：{source}，global_step={best_step}）：{resume_path}")
    if len(candidates) > 1:
        print(f"（另一個來源也有 checkpoint，但 global_step 較舊，已忽略：{candidates[1]}）")
else:
    print("沒有找到既有 checkpoint，從頭開始訓練。")


## 7. 載入模型與 LoRA

注意 chat template 用 `"qwen3"`（hybrid think 版本的 template），**不是** `"qwen3-instruct"`——
Qwen3-8B 沒有像 4B 那樣的專用非思考版本，只有 hybrid 模型，兩種 template 的 EOS/think 慣例不同，混用會造成訓練/推論格式不一致。

**已知坑**（Phase 2 實測 + 兩輪查證確認的根因鏈）：
1. Unsloth 的 `from_pretrained` 被 `_offline_aware_load` 包住，線上載入撞到「網路相關」例外會靜默重試、強制切換離線模式；
   離線模式碰到下載不完整的快取，`transformers` 不會報清楚的錯誤，而是讓 `checkpoint_files` 變成 `None`，
   炸出 `AttributeError: 'NoneType' object has no attribute 'endswith'`（清快取重試沒用，反而保證離線重試一定找不到檔案）
2. 觸發第 1 點的網路問題，根源是 huggingface_hub 新版下載大檔案預設走的 `hf_xet` 傳輸機制在 Colab 網路環境會卡死在 0%
   （不是 `hf_transfer`——那個舊機制已經在最上面「0. 下載相關環境變數」那格關掉了，但對 `hf_xet` 沒有作用）

已經在最上面把 `hf_xet` 停用（環境變數 + 移除套件雙重保險），這裡再用穩定版下載器把整包模型預先下載完整，
確保 Unsloth 之後不管走線上還是被迫走離線重試，本地都有完整檔案可用。

**已知坑（第三回合）**：就算關掉 `hf_xet`，HF 部分檔案在伺服器端本來就是用 Xet 儲存，下載連結還是會被導到
`xet-bridge` 這個 CDN；如果剛好遇到 HF 那端 CDN 簽章基礎設施異常（2026-07-13/14 這兩天已有其他人回報同樣狀況，
見 huggingface/datasets#8328），單一檔案的下載連結會直接被拒絕（`403 SignatureError: invalid key pair id`）。
這是 HF 那邊的問題、不是我們設定錯，通常是間歇性的——`snapshot_download` 本身可續傳，已經下載完的大檔案
（safetensors 權重佔了下載量的大宗）不會重抓，所以下面用重試迴圈自動處理，不用每次都手動重跑整格。

**已知坑（第四回合）**：即使 `snapshot_download` 完全下載成功，Unsloth 內部的 `_offline_aware_load` 判斷式
`_get_effective_local_files_only`（只吃 `local_files_only` kwarg 或 `HF_HUB_OFFLINE`/`TRANSFORMERS_OFFLINE`
環境變數兩種訊號）在**全新 Colab session 的第一次呼叫就會判定為離線**——排除是殘留狀態污染。權重檔在離線
模式下能正確找到快取（因為我們剛下載完），但 tokenizer 那步（`AutoConfig.from_pretrained` 找 `config.json`）
卻找不到，懷疑是 `snapshot_download`（不指定 `revision`，預設抓 `main`）跟 transformers 自己解析 `main` 版本
對應到哪個 snapshot 資料夾的機制，在離線模式下沒有完全對齊。與其繼續猜為什麼會進離線模式，改成更穩妥的做法：
呼叫 Unsloth 之前，自己先用 transformers 原生的 `AutoConfig`/`AutoTokenizer` 成功呼叫一次（走 transformers
自己標準、正常線上的解析流程），把快取用「Unsloth 之後不管線上或離線都認得出來」的正確方式寫好。


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from huggingface_hub import snapshot_download
from huggingface_hub.errors import HfHubHTTPError
from transformers import AutoConfig, AutoTokenizer
import time

# 防禦性清理：即使目前判斷這兩個環境變數不是本回合問題的根因，留著當免費的保險不會有壞處。
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

MAX_DOWNLOAD_RETRIES = 5
BASE_MODEL = cfg["model"]["base_model"]

print(f"預先下載 {BASE_MODEL}（hf_xet 已停用，走穩定版 HTTP 下載）...")
for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
    try:
        snapshot_download(repo_id=BASE_MODEL, token=HF_TOKEN)
        print("下載完成")
        break
    except HfHubHTTPError as e:
        if attempt == MAX_DOWNLOAD_RETRIES:
            raise
        wait_s = 15 * attempt
        print(
            f"下載第 {attempt} 次遇到錯誤（{type(e).__name__}: {e}）。"
            f"已下載的部分（多半是最大宗的權重檔）不會重抓，{wait_s} 秒後重試（{attempt}/{MAX_DOWNLOAD_RETRIES}）..."
        )
        time.sleep(wait_s)

# 用 transformers 原生機制自己先成功解析一次 config/tokenizer（正常線上模式），
# 確保快取是用「Unsloth 之後不管線上或離線都認得出來」的標準方式寫好，
# 不依賴 Unsloth 內部離線判斷邏輯是否跟我們的下載結果對齊。
print("預先驗證 config/tokenizer 快取...")
AutoConfig.from_pretrained(BASE_MODEL, token=HF_TOKEN)
AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
print("驗證通過，開始載入模型")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg["model"]["base_model"],
    max_seq_length=cfg["model"]["max_length"],
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    token=HF_TOKEN,
)

tokenizer = get_chat_template(tokenizer, chat_template=cfg["model"]["chat_template"])

model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora"]["r"],
    target_modules=cfg["lora"]["target_modules"],
    lora_alpha=cfg["lora"]["alpha"],
    lora_dropout=cfg["lora"]["dropout"],
    bias="none",
    use_gradient_checkpointing=cfg["lora"]["use_gradient_checkpointing"],
    random_state=cfg["lora"]["random_state"],
    use_rslora=False,
    loftq_config=None,
)


## 8. 載入資料（Phase 1 推送的 HF private dataset）

`messages` 欄位已經是 system/user/assistant 三則訊息、assistant 內容是固定 JSON 字串、**完全沒有 `<think>` 標記**
（Phase 1 `build_dataset.py` 產出時就是這樣設計的）。

`qwen3` chat template 的邏輯是：只要 assistant 內容沒有 `<think>` 標記，且是對話中最後一則 assistant 訊息，
template 會自動幫你插入空的 `<think>\n\n</think>\n\n` 再接上原本的內容——這正是 PLAN.md §4.1 設計的「訓練端一律
non-thinking 格式」的實際運作機制（查證結果，直接讀 unsloth/chat_templates.py 原始碼確認）。

**已知坑**：這裡也可能撞到跟模型下載一樣的 `xet-bridge` 簽章錯誤（HF 那邊的基礎設施問題，見上面「7. 載入模型」
那格的說明——連我們自己的 dataset repo 都會中，證實是 HF 全站範圍的問題，不是特定 repo），所以一樣包重試迴圈。


In [ ]:
from datasets import load_dataset
import time

MAX_DATASET_RETRIES = 5
for attempt in range(1, MAX_DATASET_RETRIES + 1):
    try:
        raw_dataset = load_dataset(DATASET_REPO, data_files="train_sft.jsonl", split="train", token=HF_TOKEN)
        break
    except HfHubHTTPError as e:
        if attempt == MAX_DATASET_RETRIES:
            raise
        wait_s = 15 * attempt
        print(
            f"載入 dataset 第 {attempt} 次遇到錯誤（{type(e).__name__}: {e}），"
            f"{wait_s} 秒後重試（{attempt}/{MAX_DATASET_RETRIES}）..."
        )
        time.sleep(wait_s)

print(f"train_sft.jsonl 原始筆數：{len(raw_dataset)}")

SEED = cfg["training"]["seed"]
max_train_samples = mode_cfg["max_train_samples"]
eval_holdout = mode_cfg["eval_holdout"]

pool = raw_dataset.shuffle(seed=SEED)
if max_train_samples is not None:
    pool = pool.select(range(min(max_train_samples + eval_holdout, len(pool))))

split = pool.train_test_split(test_size=eval_holdout, seed=SEED)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"[{MODE}] train={len(train_dataset)}　eval_holdout={len(eval_dataset)}（只監控 loss，不參與梯度更新，皆取自 train_sft，不動 DRCD 官方 dev）")


In [ ]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}


train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)


In [ ]:
# 自動驗證：抽 5 筆確認空 think block 真的被插入了（不是只憑肉眼看）
sample_texts = train_dataset[:5]["text"]
for t in sample_texts:
    assert "<think>\n\n</think>" in t, "空 think block 沒有被正確插入，檢查 chat_template 設定！"
print("驗證通過：抽樣的訓練樣本都包含空 think block（non-thinking 格式正確）")
print("=" * 60)
print("完整渲染範例（第一筆）：")
print(sample_texts[0])


## 9. 建立 Trainer

`hub_strategy="checkpoint"` 會把最新 checkpoint 推到 Hub 上固定命名的 `last-checkpoint/` 子資料夾（`every_save` 的行為
再加這個）。`DriveBackupCallback` 是額外的次要備援通道（PLAN.md 決策：HF Hub 主力 + Drive 備援），
在本地 checkpoint 完整寫完之後才複製到 Drive，不會有 FUSE 半寫入毀檔的風險。


In [ ]:
import shutil
from transformers import TrainerCallback


class DriveBackupCallback(TrainerCallback):
    """本地 checkpoint 存完之後，複製一份到 Google Drive 當次要備援（HF Hub 是主力）。"""

    def __init__(self, drive_backup_dir: str):
        self.drive_backup_dir = drive_backup_dir

    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if os.path.isdir(ckpt_dir):
            dest = os.path.join(self.drive_backup_dir, "last-checkpoint")
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(ckpt_dir, dest)
            print(f"[drive-backup] 已備份 checkpoint-{state.global_step} 到 {dest}")
        return control


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=cfg["model"]["max_length"],
    per_device_train_batch_size=cfg["training"]["per_device_train_batch_size"],
    gradient_accumulation_steps=cfg["training"]["gradient_accumulation_steps"],
    num_train_epochs=mode_cfg["num_train_epochs"],
    learning_rate=cfg["training"]["learning_rate"],
    lr_scheduler_type=cfg["training"]["lr_scheduler_type"],
    warmup_ratio=cfg["training"]["warmup_ratio"],
    optim=cfg["training"]["optim"],
    bf16=cfg["training"]["bf16"],
    weight_decay=cfg["training"]["weight_decay"],
    packing=cfg["training"]["packing"],
    seed=cfg["training"]["seed"],
    logging_steps=mode_cfg["logging_steps"],
    eval_strategy="steps",
    eval_steps=mode_cfg["eval_steps"],
    save_strategy="steps",
    save_steps=mode_cfg["save_steps"],
    save_total_limit=cfg["training"]["save_total_limit"],
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="checkpoint",
    hub_private_repo=True,
    report_to="wandb",
    run_name=f"{MODE}-{HUB_MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_args,
    callbacks=[DriveBackupCallback(DRIVE_BACKUP_DIR)],
)

# 只對 assistant 回應算 loss，不浪費在 system/user 的長文章 context 上（Unsloth 現行版本會自動抓 template 邊界）
trainer = train_on_responses_only(trainer)


## 10. 開始訓練（自動接續前一次的 checkpoint，如果有的話）

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=resume_path)
print(trainer_stats)


## 11. 訓練後收尾

`.train()` 中途只有每個 `save_steps` 才會推送，訓練結束的當下不一定剛好落在存檔點上，
所以這裡明確再 push 一次，確保 Hub 上是最終狀態。


In [ ]:
trainer.push_to_hub(commit_message="training complete")
print(f"最終 adapter 已推送：https://huggingface.co/{HUB_MODEL_ID}")
print(f"W&B run：{wandb.run.url if wandb.run else '(未啟用)'}")


## 12.（Phase 2 專用）斷線續訓演練步驟

1. 先跑完上面所有 cell 一次，確認 sanity check 訓練有正常跑起來、loss 有在下降（看 W&B 或 log）
2. **重新執行一次**：直接從頭 Runtime > Run all，讓它重新開始訓練
3. 訓練跑到中途（例如看到 log 印出 global step 到 10 幾左右）時，手動 **Runtime > Interrupt execution**
   （模擬 Colab Pro 斷線）
4. 確認此時 HF Hub 的 checkpoint repo（`HUB_MODEL_ID`）或本地 `outputs/` 底下已經有 `checkpoint-N` 資料夾
   （代表至少存過一次檔）
5. **不要清除任何 state，直接 Runtime > Run all 重跑全部 cell**
6. 觀察「續訓偵測」那一格的輸出，應該要印出「偵測到可續訓的 checkpoint」，而不是「從頭開始訓練」
7. 訓練開始後，觀察 log 的 global step 是否**接著中斷前的進度**繼續（而不是從 0 重來）
8. 全部跑完後回報：是否成功接續、global step 有沒有對上、loss 曲線在 W&B 上是否連續（同一個 run 或至少邏輯銜接）

驗收標準（PLAN.md Phase 2）：以上演練成功 = 斷線續訓機制驗證通過，可以進入 Phase 3 全量訓練。
